# Can we predict next-month arrears?
Target: `arrears_next_month`. Each row is one account-month (1,375 accounts, Jul 2023 – Dec 2025).

Plan: inspect data → decide which columns are legitimately available at prediction time → time-based split → compare baselines and models → report.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values(['billing_month','meter_ref']).reset_index(drop=True)
print(df.shape, 'missing:',int(df.isna().sum().sum()), 'dup account-months:',int(df.duplicated(['meter_ref','billing_month']).sum()))
df.describe().T

In [ ]:
print('Base rate of arrears:', round(df.arrears_next_month.mean(),3))
fig,ax=plt.subplots(1,3,figsize=(15,3.5))
df.groupby('late_payments_12m').arrears_next_month.mean().plot.bar(ax=ax[0],title='Arrears rate by late payments (12m)')
df.groupby('tariff').arrears_next_month.mean().plot.bar(ax=ax[1],title='Arrears rate by tariff')
df.groupby('billing_month').arrears_next_month.mean().plot(ax=ax[2],title='Arrears rate over time')
plt.tight_layout();plt.show()

## Leakage check: `final_notice_sent`
The dictionary says this is 1 if a final notice was sent *in the billing cycle that followed* the billing month, i.e. **after** the point where we would make the prediction. It is also almost a restatement of the target:

In [ ]:
print(df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size']))
print('AUC of final_notice_sent alone:', round(roc_auc_score(df.arrears_next_month, df.final_notice_sent),3))

A final notice only goes out once an account is already behind, so using it would be predicting arrears from a consequence of arrears. **I exclude it from the real model** and show it below only to quantify how misleading it would be.

## Features and split
Only information known at the end of the billing month is used. I add `kwh_per_occupant` and `eur_per_kwh`. The split is **temporal** (train on earlier months, test on the last 6), because the model would be used going forward, and accounts repeat across months, so a random row split would put the same account in train and test and flatter the scores.

In [ ]:
df['kwh_per_occupant']=df.kwh/df.occupants
df['eur_per_kwh']=df.bill_eur/df.kwh
num=['occupants','kwh','bill_eur','late_payments_12m','kwh_per_occupant','eur_per_kwh']
cat=['tariff']
cut=pd.Timestamp('2025-07-01')
tr,te=df[df.billing_month<cut],df[df.billing_month>=cut]
print(len(tr),len(te),'| test rows whose account is also in train:',round(te.meter_ref.isin(tr.meter_ref).mean(),2))
print('base rate train/test:',round(tr.arrears_next_month.mean(),3),round(te.arrears_next_month.mean(),3))
y_tr,y_te=tr.arrears_next_month,te.arrears_next_month

In [ ]:
def prep(nums,scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',nums),('c',OneHotEncoder(),cat)])
def gbm(cols): return make_pipeline(prep(cols,False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))
models={
 'logistic regression':make_pipeline(prep(num,True),LogisticRegression(max_iter=1000)),
 'gradient boosting':gbm(num),
}
probs={'late_payments only (rule)':te.late_payments_12m.values/5}
for k,m in models.items():
    probs[k]=m.fit(tr[num+cat],y_tr).predict_proba(te[num+cat])[:,1]
res=pd.DataFrame({k:dict(ROC_AUC=roc_auc_score(y_te,p),PR_AUC=average_precision_score(y_te,p)) for k,p in probs.items()}).T.round(3)
res.loc['(no skill)']=[0.5,round(y_te.mean(),3)]
res

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for k in ['logistic regression','gradient boosting']:
    p,r,_=precision_recall_curve(y_te,probs[k]); ax[0].plot(r,p,label=k)
ax[0].axhline(y_te.mean(),ls='--',c='gray',label='base rate');ax[0].set(xlabel='recall',ylabel='precision',title='Precision-recall (test)');ax[0].legend()
for k in ['logistic regression','gradient boosting']:
    f,m=calibration_curve(y_te,probs[k],n_bins=8,strategy='quantile'); ax[1].plot(m,f,'o-',label=k)
ax[1].plot([0,.6],[0,.6],'k--');ax[1].set(xlabel='predicted',ylabel='observed',title='Calibration (test)');ax[1].legend()
plt.tight_layout();plt.show()
for k in ['logistic regression','gradient boosting']: print(k,'Brier',round(brier_score_loss(y_te,probs[k]),4),'vs constant',round(brier_score_loss(y_te,np.full(len(y_te),y_tr.mean())),4))

## What drives it, and what is a usable operating point?

In [ ]:
lr=models['logistic regression']
print(pd.Series(lr[-1].coef_[0],index=lr[0].get_feature_names_out()).round(3).sort_values())
t=pd.DataFrame({'p':probs['gradient boosting'],'y':y_te.values})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=False)+1
print(t.groupby('decile').y.agg(['mean','size']).round(3).rename(columns={'mean':'arrears_rate'}))
print('Top 20% of scores contain', round(t[t.decile>=9].y.sum()/t.y.sum(),2),'of all arrears cases')

## Cost of the leaky feature (for illustration only)

In [ ]:
cols=num+['final_notice_sent']
pl=gbm(cols).fit(tr[cols+cat],y_tr).predict_proba(te[cols+cat])[:,1]
print('With final_notice_sent  ROC AUC %.3f  PR AUC %.3f'%(roc_auc_score(y_te,pl),average_precision_score(y_te,pl)))
print('Without (honest)        ROC AUC %.3f  PR AUC %.3f'%tuple(res.loc['gradient boosting']))

## Why is the test base rate 3x the training rate? (distribution shift)
The arrears rate is ~2–4% for 18 months, then climbs from early 2025 to ~26% by Dec 2025. The average bill jumps ~14% in Jan 2025 at unchanged kWh, which points to a price/tariff increase, and arrears follow it with a lag. The inputs we have don't otherwise change (late payments, usage), so the model can't see this regime directly.

In [ ]:
g=df.groupby('billing_month').agg(arrears_rate=('arrears_next_month','mean'),avg_bill=('bill_eur','mean'),avg_kwh=('kwh','mean'),eur_per_kwh=('eur_per_kwh','mean'))
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
g.arrears_rate.plot(ax=ax[0],title='Arrears rate by billing month');g[['eur_per_kwh']].plot(ax=ax[1],title='Average EUR per kWh')
plt.tight_layout();plt.show()

## Conclusions
**Yes, partly.** Using only information available at billing time, the models rank accounts well above chance, but this is not a high-accuracy predictor.

* Held-out last 6 months (Jul–Dec 2025): ROC AUC about 0.74 and PR AUC about 0.49 against a no-skill 0.22 (see the results table; the late-payments-only rule gets 0.70 / 0.40). Logistic regression and gradient boosting are about equal, so the signal is simple: more late payments, higher bill, fewer occupants, higher price per kWh.
* Useful for prioritising: the top-scored decile has ~61% arrears and the top 20% holds ~46% of all arrears cases.
* **`final_notice_sent` is excluded on purpose.** It is recorded after the billing month and mostly follows arrears. Including it gives AUC ~0.97, which would not hold in real use.
* **The arrears rate is not stable.** It rose from ~3% to ~26% during 2025 after an apparent price rise in Jan 2025. Probabilities are therefore poorly calibrated on the test period (Brier barely beats a constant guess), and there are only 6–12 post-change months to learn from. Use the scores for ranking, and recalibrate or retrain on recent months before relying on absolute probabilities.

Caveats: single temporal split, no hyperparameter tuning, no per-account history features (previous arrears, payment trend), which are the obvious next step if that data exists.